In [1]:
import cv2
import mediapipe as mp
import numpy as np
import math

# Initialize camera
cap = cv2.VideoCapture(0)

# MediaPipe hand setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1)
mp_draw = mp.solutions.drawing_utils

# Calculator buttons
buttons = [
["7","8","9","/"],
["4","5","6","*"],
["1","2","3","-"],
["0",".","=","+"]
]

equation = ""
clicked_button = None

# Draw calculator buttons
def draw_buttons(img):

    for i in range(len(buttons)):
        for j in range(len(buttons[i])):

            x = 100*j + 50
            y = 100*i + 50

            cv2.rectangle(img,(x,y),(x+80,y+80),(200,200,200),cv2.FILLED)

            cv2.putText(img, buttons[i][j],
                        (x+25,y+55),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1,(0,0,0),2)

while True:

    success, img = cap.read()
    img = cv2.flip(img,1)

    imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(imgRGB)

    lm_list = []
    x2, y2 = 0,0

    if results.multi_hand_landmarks:

        for handLms in results.multi_hand_landmarks:

            mp_draw.draw_landmarks(img, handLms, mp_hands.HAND_CONNECTIONS)

            for id, lm in enumerate(handLms.landmark):

                h,w,c = img.shape
                cx,cy = int(lm.x*w), int(lm.y*h)

                lm_list.append([id,cx,cy])

            if len(lm_list) >= 9:
                x2,y2 = lm_list[8][1], lm_list[8][2]

                cv2.circle(img,(x2,y2),10,(255,0,255),cv2.FILLED)

    # Draw calculator
    draw_buttons(img)

    # Detect button touch
    for i in range(len(buttons)):
        for j in range(len(buttons[i])):

            x = 100*j + 50
            y = 100*i + 50

            if x < x2 < x+80 and y < y2 < y+80:
                clicked_button = buttons[i][j]

    # Update equation
    if clicked_button is not None:

        if clicked_button == "=":
            try:
                equation = str(eval(equation))
            except:
                equation = "Error"
        else:
            equation += clicked_button

        clicked_button = None

    # Display result box
    cv2.rectangle(img,(50,420),(450,480),(50,50,50),cv2.FILLED)

    cv2.putText(img, equation,
                (60,460),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,(255,255,255),2)

    # Show window
    cv2.imshow("Gesture Scientific Calculator", img)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()